# 01 — Cross-matching: MAXI-SIMBAD prefilter, BAT-MAXI-SIMBAD-ZTF, and compact-object classification

Report sections 3.1 and 3.2. Three stages, in dependency order:

1. **Step 1 (`maxi_simbad_prefilter`)** — cross-match the full MAXI all-sky source list against
   SIMBAD, keeping XRB-typed sources → `step1_maxi_xrb_sources.csv`.
2. **Step 1b (`classify_compact_objects`)** — classify each of those as BH/NS and
   persistent/transient using BlackCAT + Liu et al. → `step1b_maxi_xrb_classified.csv`.
   This feeds notebook 02 (outburst detection).
3. **BAT-MAXI-SIMBAD-ZTF (`bat_maxi_simbad_ztf`)** — a separate, narrower sample restricted to
   Swift/BAT sources with a confirmed MAXI + SIMBAD + ZTF match (report §3.1's "Gold" sample).

See `docs/algorithms.md` §1 for the matching logic and `docs/report.pdf` §3 for the physics.

In [ ]:
import sys
sys.path.insert(0, '../src')

from xrb_pipeline.crossmatch import maxi_simbad_prefilter as prefilter
from xrb_pipeline.crossmatch import classify_compact_objects as cls
from xrb_pipeline.crossmatch import bat_maxi_simbad_ztf as xm

## Step 1 — MAXI-SIMBAD prefilter

Downloads MAXI's all-sky source list and cone-searches SIMBAD around every position
(2 arcmin), keeping `otype in {XB, LXB, HXB}`. Takes a few minutes — one polite,
rate-limited SIMBAD query per MAXI source.

In [ ]:
# xrb_df = prefilter.run_pipeline(output_csv="step1_maxi_xrb_sources.csv")
print("Uncomment above to run the full prefilter (takes a while).")

SIMBAD sometimes types a known XRB as generic `'X'`, or hasn't classified a brand-new
transient yet — patch the catalogue by hand here before moving on.

In [ ]:
# xrb_df = prefilter.apply_manual_corrections(
#     xrb_df,
#     additions=[{"maxi_name": "MAXI J1820+070", "maxi_ra": 275.091, "maxi_dec": 7.185,
#                 "simbad_name": "MAXI J1820+070", "simbad_otype": "LXB",
#                 "simbad_ra": "18 20 21.9", "simbad_dec": "+07 11 07"}],
#     removals=[],
# )
# xrb_df.to_csv("step1_maxi_xrb_sources.csv", index=False)
print("Uncomment and edit only if SIMBAD missed or mis-typed something you know about.")

## Step 1b — Compact-object classification

In [ ]:
INPUT_CSV = "step1_maxi_xrb_sources.csv"

# df_in = pd.read_csv(INPUT_CSV)
# df_classified = cls.classify_sources(df_in)
# cls.print_summary(df_classified)
# df_classified.to_csv("step1b_maxi_xrb_classified.csv", index=False)
print("Uncomment above once step1_maxi_xrb_sources.csv is ready.")

Output: `step1b_maxi_xrb_classified.csv`, with `compact_object` and `persistent_flag`
columns. This is the input to notebook `02_outburst_detection.ipynb`.

## BAT → MAXI → SIMBAD → ZTF (narrow "Gold" sample)

Separate from the two steps above — requires a Swift/BAT catalog FITS file. Each step
writes an intermediate CSV so the run can be resumed.

In [ ]:
BAT_FITS = "BAT_catalog.fits"   # <- point this at your BAT catalog
OUTDIR = "../data/crossmatch"

# xrb_df = xm.run_pipeline(BAT_FITS, OUTDIR)
# xm.print_funnel_summary(BAT_FITS, OUTDIR)
print("Uncomment above once BAT_FITS points at a real catalog file.")